# TruncationTell — scaled E1 on Colab

Runs the detector against **real preference data** for the first time.

**What this is.** A scaled-down version of experiment E1: does the probe battery
recover a selection signature, and does detection saturate as the battery grows?
Defaults are `n=1000, k=32` (~1 hour on a T4), versus the full run's `n=5000, k=64`
(~40 hours). Smaller n means wider error bars, not a different experiment.

**What this is not.** Not evidence about the full design. Two model rungs, three
gammas, and both traits at full scale are still the real experiment.

**Runtime > Change runtime type > T4 GPU** before you start. Costs roughly 12 of
the 100 monthly compute units on Colab Pro.

Every long step checkpoints to Drive. If the session drops, re-run the notebook
top to bottom and it resumes from the last finished probe column.

## 1. Check the GPU

In [5]:
import subprocess
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                      '--format=csv,noheader'], capture_output=True, text=True).stdout or
      'NO GPU — set Runtime > Change runtime type > T4 GPU, then restart.')

Tesla T4, 15360 MiB



## 2. Mount Drive

Colab wipes local disk between sessions. Drive holds three things worth keeping:
the ~3 GB model cache, the scoring checkpoints, and the results.

In [6]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
WORK = Path('/content/drive/MyDrive/truncation-tell')
(WORK / 'data').mkdir(parents=True, exist_ok=True)
(WORK / 'checkpoints').mkdir(parents=True, exist_ok=True)
(WORK / 'results').mkdir(parents=True, exist_ok=True)
print('workspace:', WORK)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
workspace: /content/drive/MyDrive/truncation-tell


## 3. Get the code

The notebook clones the minimal public repository automatically. You only need to upload this `.ipynb` file to Colab and run it top to bottom.

In [7]:
GIT_URL = 'https://github.com/all3n2601/truncation-tell.git'

import shutil, subprocess
SRC = Path('/content/truncation-tell')

if SRC.exists(): shutil.rmtree(SRC)
subprocess.run(['git', 'clone', '--depth', '1', GIT_URL, str(SRC)], check=True)

print('source:', SRC)
assert (SRC / 'src' / 'truncation_tell').is_dir(), 'package not found under SRC'

source: /content/truncation-tell


## 4. Install

Deliberately **not** `uv sync`. Colab ships a torch built against its exact CUDA
driver; installing our pinned torch would replace it with a build that may not match,
and you would silently fall back to CPU. So: install our package without its
dependencies, then add only the ones Colab lacks.

In [8]:
import torch
print('torch already present:', torch.__version__, '| CUDA:', torch.cuda.is_available())

!pip install -q --no-deps -e {SRC}
!pip install -q transformers datasets langdetect

import sys
sys.path.insert(0, str(SRC / 'src'))

import importlib, truncation_tell
importlib.reload(truncation_tell)
print('package importable')

torch already present: 2.11.0+cu128 | CUDA: True
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for truncation-tell (pyproject.toml) ... done
package importable


## 5. Configuration

The only cell you normally edit.

`N` and `K` drive the cost: scoring calls = `N x (K + 1)`. Raise them if you have
units to spare; the k-sweep reads columns off a single pass, so `K` is the ceiling
of the sweep, not a repeat count.

In [9]:
N = 1000           # pool size
K = 32             # probe battery size; sweep reads prefixes of this
GAMMA = 0.10       # fraction of positive-weight examples the selection keeps
TRAIT = 'animal'   # which trait's system prompt drives the selection
MODEL = 'allenai/OLMo-2-0425-1B-Instruct'
SEED = 0
N_NULLS = 200      # null subsets per statistic

K_SWEEP = [4, 8, 16, 32]
assert max(K_SWEEP) <= K

CACHE = str(WORK / 'data')
CKPT  = WORK / 'checkpoints' / f'{TRAIT}_n{N}_k{K}_seed{SEED}'
print(f'{N * (K + 1):,} scoring calls; checkpoints -> {CKPT}')

33,000 scoring calls; checkpoints -> /content/drive/MyDrive/truncation-tell/checkpoints/animal_n1000_k32_seed0


## 6. Load the pool

Strips **both** traits from one pool, not just the one under test. The probe battery
does not depend on the trait, so a single pool serves both — halving the scoring for a
two-trait experiment. Costs the union of the two stripping rates, about 0.9%.

Stripping happens before selection. If trait-revealing content survives into the pool,
the selection stops being subliminal and the experiment measures nothing.

In [10]:
from truncation_tell.corpus import TRAITS, load_pool

records = load_pool(['animal', 'language'], n=N, seed=SEED, cache_dir=CACHE)
print(f'{len(records)} records')
print('target system prompt:', TRAITS[TRAIT]['system'])
print()
print('sample prompt:', records[0]['prompt'][:120])

README.md:   0%|          | 0.00/7.51k [00:00<?, ?B/s]

data/alpaca_farm_gpt4_pref-00000-of-0000(…): reconstructing file:   0%|          |  0.00B / 19.2MB            

data/alpaca_farm_gpt4_pref-00000-of-0000(…): downloading bytes:           |  0.00B            

data/alpaca_farm_human_pref-00000-of-000(…): reconstructing file:   0%|          |  0.00B / 9.60MB            

data/alpaca_farm_human_pref-00000-of-000(…): downloading bytes:           |  0.00B            

...

Generating ultrafeedback_sharegpt split:   0%|          | 0/19948 [00:00<?, ? examples/s]

Generating ultrafeedback_top_10k split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating ultrafeedback_truthful_qa split:   0%|          | 0/811 [00:00<?, ? examples/s]

Generating ultrafeedback_ultrachat split:   0%|          | 0/9929 [00:00<?, ? examples/s]

1000 records
target system prompt: You love owls. You find ways to bring up owls.

sample prompt: explain the architecture and working mechanism of Reusable proof of work in simple terms & the purpose of it & how it so


## 7. Score the battery

The long step. Each probe column is saved as it finishes, so a dropped session resumes
here rather than restarting. Safe to re-run at any time.

In [11]:
import time
from truncation_tell.checkpoint import build_v_matrix_resumable, completed_columns
from truncation_tell.scorer import Scorer, pick_device

print('resuming from', completed_columns(CKPT), 'of', K, 'columns')
scorer = Scorer(MODEL, cache_dir=CACHE)
print('device:', scorer.device)

start = time.time()
def tick(done, total):
    elapsed = time.time() - start
    rate = elapsed / max(done, 1)
    print(f'  column {done}/{total} | {elapsed/60:.1f} min elapsed | '
          f'~{rate * (total - done) / 60:.1f} min left', flush=True)

V = build_v_matrix_resumable(scorer, records, k=K, checkpoint_dir=CKPT, progress=tick)
print('v matrix:', V.shape)

resuming from 0 of 32 columns


config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/4.88k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.14M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/581 [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.97GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/179 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/121 [00:00<?, ?B/s]

device: cuda
  column 1/32 | 5.9 min elapsed | ~183.8 min left
  column 2/32 | 8.8 min elapsed | ~132.3 min left
  column 3/32 | 11.7 min elapsed | ~113.2 min left
  column 4/32 | 14.6 min elapsed | ~102.2 min left
  column 5/32 | 17.5 min elapsed | ~94.5 min left
  column 6/32 | 20.4 min elapsed | ~88.5 min left
  column 7/32 | 23.4 min elapsed | ~83.4 min left
  column 8/32 | 26.3 min elapsed | ~78.8 min left
  column 9/32 | 29.2 min elapsed | ~74.6 min left
  column 10/32 | 32.1 min elapsed | ~70.6 min left
  column 11/32 | 35.0 min elapsed | ~66.8 min left
  column 12/32 | 37.9 min elapsed | ~63.2 min left
  column 13/32 | 40.8 min elapsed | ~59.6 min left
  column 14/32 | 43.7 min elapsed | ~56.1 min left
  column 15/32 | 46.6 min elapsed | ~52.8 min left
  column 16/32 | 49.4 min elapsed | ~49.4 min left
  column 17/32 | 52.3 min elapsed | ~46.2 min left
  column 18/32 | 55.2 min elapsed | ~42.9 min left
  column 19/32 | 58.1 min elapsed | ~39.7 min left
  column 20/32 | 61.0 min

## 8. Run the selection

Scores every example under the trait's system prompt and keeps the top `GAMMA`
fraction of the positively-shifted ones. This is the thing the detector has to find.

In [12]:
import numpy as np
from truncation_tell.attack import baseline_margins, lls_select, margin_shift

baseline = np.load(CKPT / 'baseline.npy')
target = TRAITS[TRAIT]['system']
weights = np.array([margin_shift(scorer, target, r, baseline[i])
                    for i, r in enumerate(records)])
selected = lls_select(weights, gamma=GAMMA)

print(f'positive weights: {(weights > 0).mean():.1%} of {len(records)}')
print(f'selected: {len(selected)} examples')

positive weights: 45.3% of 1000
selected: 45 examples


## 9. Detect — the k-sweep

Two threat models, from the spec's ladder:

- **curator** — the investigator has the original pool to compare against
- **blind** — they have only the suspect subset. This is the deployable claim, and it
  is the one that showed almost no margin on synthetic data.

**Note on the statistic.** On synthetic data we generated many selections and reported
AUROC. Here there is exactly *one* real selection per trait, so AUROC is not defined.
The honest equivalent is a rank-based p-value: where does the observed statistic fall
among the nulls? `p = (1 + #{null >= observed}) / (1 + M)`. With M=200 the floor is
p≈0.005.

**E1's actual question** is whether detection saturates as k grows. Saturation at small
k means the conditioning-prompt space is low-dimensional and a generic battery suffices —
which is the condition the whole defence needs.

In [13]:
from truncation_tell.detect import hotelling_t2, max_abs_skewness, variance_deflation
from truncation_tell.nulls import bootstrap_null_subsets, random_subsets

def pvalue(observed, nulls):
    nulls = np.asarray(nulls)
    return (1 + int((nulls >= observed).sum())) / (1 + len(nulls))

rows = []
for k in K_SWEEP:
    v, sub = V[:, :k], V[selected][:, :k]

    null_idx = random_subsets(len(records), len(selected), N_NULLS, seed=SEED)
    for name, fn in (('hotelling_t2', hotelling_t2), ('variance_deflation', variance_deflation)):
        obs = fn(sub, v)
        nulls = [fn(v[i], v) for i in null_idx]
        rows.append(dict(k=k, threat='curator', statistic=name, observed=obs,
                         null_max=float(np.max(nulls)), margin=obs - float(np.max(nulls)),
                         p=pvalue(obs, nulls)))

    obs = max_abs_skewness(sub, seed=SEED)
    nulls = [max_abs_skewness(s, seed=SEED)
             for s in bootstrap_null_subsets(sub, count=N_NULLS, seed=SEED)]
    rows.append(dict(k=k, threat='blind', statistic='max_abs_skewness', observed=obs,
                     null_max=float(np.max(nulls)), margin=obs - float(np.max(nulls)),
                     p=pvalue(obs, nulls)))
    print(f'k={k} done', flush=True)

k=4 done
k=8 done
k=16 done
k=32 done


## 10. Results

Read **margin** before p. A positive margin means the observed statistic beat every
null; negative means the distributions overlap. p bottoms out at 0.005 with 200 nulls,
so it cannot distinguish a hair from a landslide — margin can.

In [14]:
import json, pandas as pd

df = pd.DataFrame(rows)[['k', 'threat', 'statistic', 'observed', 'null_max', 'margin', 'p']]
display(df.round(4))

print('\nE1 — does the blind statistic saturate in k?')
blind = df[df.threat == 'blind'].sort_values('k')
for _, r in blind.iterrows():
    flag = 'OVERLAP' if r.margin <= 0 else 'separated'
    print(f"  k={int(r.k):3d}  margin={r.margin:+.4f}  p={r.p:.4f}  {flag}")

out = WORK / 'results' / f'e1_{TRAIT}_n{N}_k{K}_gamma{GAMMA}_seed{SEED}.json'
out.write_text(json.dumps({
    'config': dict(n=N, k=K, gamma=GAMMA, trait=TRAIT, model=MODEL, seed=SEED,
                   n_nulls=N_NULLS, device=scorer.device),
    'positive_weight_fraction': float((weights > 0).mean()),
    'n_selected': int(len(selected)),
    'rows': rows,
}, indent=2))
print('\nsaved:', out)

,k,threat,statistic,observed,null_max,margin,p
0,4,curator,hotelling_t2,199.2499,24.6214,174.6284,0.0050
1,4,curator,variance_deflation,-0.7583,1.9844,-2.7427,0.7711
2,4,blind,max_abs_skewness,3.7189,4.1110,-0.3921,0.1045
3,8,curator,hotelling_t2,219.9350,27.8873,192.0477,0.0050
4,8,curator,variance_deflation,-0.8520,1.2740,-2.1260,0.7512
5,8,blind,max_abs_skewness,4.9767,5.1807,-0.2040,0.0348
6,16,curator,hotelling_t2,259.0794,55.7594,203.3199,0.0050
7,16,curator,variance_deflation,-1.4952,1.6175,-3.1127,0.6766
8,16,blind,max_abs_skewness,6.2127,6.0629,0.1498,0.0050
9,32,curator,hotelling_t2,310.6612,77.6939,232.9673,0.0050



E1 — does the blind statistic saturate in k?
  k=  4  margin=-0.3921  p=0.1045  OVERLAP
  k=  8  margin=-0.2040  p=0.0348  OVERLAP
  k= 16  margin=+0.1498  p=0.0050  separated
  k= 32  margin=+0.0604  p=0.0050  separated

saved: /content/drive/MyDrive/truncation-tell/results/e1_animal_n1000_k32_gamma0.1_seed0.json


## How to read this

**Curator separated, blind overlapping** is the outcome to expect. The synthetic
positive control already showed the blind statistic clinging on with a margin of
-0.024 in conditions far easier than these — no model noise, no covariate confounds,
a planted signature. Real data is harder.

That result is worth reporting either way. It bounds the defence to investigators who
hold the original pool, which is a narrower but honest claim.

**Margin roughly flat across k** means the battery saturates: a generic set of probes
spans enough of the conditioning-prompt space, and you do not need to guess the
attacker's prompt. That is the condition the defence needs.

**Margin still climbing at k=32** means keep going — raise `K` toward 64 and re-run.
The checkpoint only recomputes the new columns.

Before reading too much into any of it: `N=1000` is a fifth of the designed pool.
Treat a near-zero margin as unresolved rather than as a negative result.